# Notebook 09: Advanced Analytics & Statistical Tests

## Project: Indian Air Quality Index (AQI) Comprehensive Analysis
## BTech Final Year Project - Data Science & Machine Learning (8th Semester)

### Objective:
Perform advanced statistical testing, hypothesis testing, PCA (Principal Component Analysis), clustering analysis, and advanced pattern discovery.

### Prerequisites:
- Complete Notebook 01 (generates city_day_cleaned.csv)
- Libraries: pandas, numpy, scipy, scikit-learn, statsmodels

### Run Time: 20-30 minutes

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import ttest_ind, f_oneway, chi2_contingency, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported!')

### Explanation:

- **from scipy.stats import ttest_ind**: Independent t-test for comparing two groups.

- **from scipy.stats import f_oneway**: One-way ANOVA for comparing three or more groups.

- **from scipy.stats import chi2_contingency**: Chi-square test for categorical variable independence.

- **from sklearn.decomposition import PCA**: Principal Component Analysis for dimensionality reduction.

- **from sklearn.cluster import KMeans**: K-Means clustering algorithm for grouping similar data points.

In [ ]:
data_path = os.path.join('..', 'datasets', 'city_day_cleaned.csv')
df = pd.read_csv(data_path, parse_dates=['Datetime'])
print(f'Loaded {df.shape[0]} rows')

## Step 2: Hypothesis Testing - T-Test (Winter vs Summer)

In [ ]:
df['Month'] = df['Datetime'].dt.month
winter_aqi = df[df['Month'].isin([12, 1, 2])]['AQI']
summer_aqi = df[df['Month'].isin([3, 4, 5])]['AQI']
t_stat, p_value = ttest_ind(winter_aqi.dropna(), summer_aqi.dropna())
print('T-Test: Winter vs Summer AQI')
print(f'T-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'Result: {"SIGNIFICANT difference" if p_value < 0.05 else "NO significant difference"}')

### Explanation:

- **ttest_ind()**: Independent two-sample t-test compares means of two groups.

- **Null hypothesis**: Both groups have same mean.

- **p-value < 0.05**: Reject null hypothesis (groups are significantly different).

- **Winter months**: December, January, February.

- **Summer months**: March, April, May.

## Step 3: ANOVA Test (Regional Differences)

In [ ]:
regions = df['Region'].unique()
region_groups = [df[df['Region'] == r]['AQI'].dropna() for r in regions]
f_stat, p_value = f_oneway(*region_groups)
print('One-Way ANOVA: Regional AQI Differences')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'Result: {"SIGNIFICANT differences among regions" if p_value < 0.05 else "NO significant differences"}')

### Explanation:

- **f_oneway()**: One-way ANOVA tests if three or more groups have different means.

- **F-statistic**: Ratio of between-group variance to within-group variance.

- **p-value < 0.05**: At least one region is significantly different from others.

## Step 4: Chi-Square Test (AQI Category vs Region)

In [ ]:
contingency_table = pd.crosstab(df['Region'], df['AQI_Bucket'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print('Chi-Square Test: AQI Category vs Region')
print(f'Chi-square statistic: {chi2:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'Degrees of freedom: {dof}')
print(f'Result: {"Variables are DEPENDENT" if p_value < 0.05 else "Variables are INDEPENDENT"}')

### Explanation:

- **pd.crosstab()**: Creates contingency table (frequency count of category combinations).

- **chi2_contingency()**: Chi-square test for independence between categorical variables.

- **Null hypothesis**: Region and AQI category are independent.

- **p-value < 0.05**: Variables are dependent (region affects AQI category).

## Step 5: Principal Component Analysis (PCA)

In [ ]:
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3']
df_pca = df[pollutants].dropna()
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_pca)
pca = PCA()
pca_result = pca.fit_transform(scaled_data)
explained_var = pca.explained_variance_ratio_
print('PCA Results:')
for i, var in enumerate(explained_var[:5]):
    print(f'PC{i+1}: {var*100:.2f}% variance explained')

### Explanation:

- **PCA()**: Transforms correlated features into uncorrelated principal components.

- **explained_variance_ratio_**: Percentage of total variance captured by each component.

- **PC1**: First principal component captures most variance.

- **Purpose**: Dimensionality reduction while preserving information.

## Step 6: Scree Plot (PCA)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_var)+1), explained_var*100, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=90, color='r', linestyle='--', label='90% Variance Threshold')
plt.xlabel('Principal Component', fontsize=12, fontweight='bold')
plt.ylabel('Explained Variance (%)', fontsize=12, fontweight='bold')
plt.title('Scree Plot - PCA Explained Variance', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: PCA 2D Visualization

In [ ]:
df['Season'] = df['Datetime'].dt.month.map({12:'Winter', 1:'Winter', 2:'Winter', 3:'Summer', 4:'Summer', 5:'Summer', 6:'Monsoon', 7:'Monsoon', 8:'Monsoon', 9:'Post-Monsoon', 10:'Post-Monsoon', 11:'Post-Monsoon'})
df_sample = df.dropna(subset=pollutants).sample(n=5000, random_state=42)
scaled_sample = scaler.transform(df_sample[pollutants])
pca_2d = PCA(n_components=2).fit_transform(scaled_sample)
plt.figure(figsize=(10, 8))
for season in ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']:
    mask = df_sample['Season'] == season
    plt.scatter(pca_2d[mask, 0], pca_2d[mask, 1], label=season, alpha=0.5, s=20)
plt.xlabel('PC1', fontsize=12, fontweight='bold')
plt.ylabel('PC2', fontsize=12, fontweight='bold')
plt.title('PCA 2D Projection - Seasonal Clustering', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Explanation:

- **n_components=2**: Reduces to 2 dimensions for visualization.

- **Scatter plot**: Shows data points in PC1-PC2 space colored by season.

- **Clusters**: Similar seasons should group together.

## Step 8: K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean = df.dropna(subset=pollutants)
scaled_clean = scaler.fit_transform(df_clean[pollutants])
clusters = kmeans.fit_predict(scaled_clean)
df_clean['Cluster'] = clusters
print(f'Cluster distribution:')
print(df_clean['Cluster'].value_counts().sort_index())

### Explanation:

- **KMeans(n_clusters=3)**: Groups data into 3 clusters based on pollutant profiles.

- **random_state=42**: Ensures reproducible clustering.

- **n_init=10**: Runs algorithm 10 times with different initializations (picks best).

- **fit_predict()**: Fits model AND assigns cluster labels.

## Step 9: Cluster Analysis

In [ ]:
cluster_stats = df_clean.groupby('Cluster')[pollutants + ['AQI']].mean()
print('Cluster Statistics (Mean Values):')
cluster_stats.round(1)

## Step 10: Elbow Method (Optimal K)

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(scaled_clean)
    inertias.append(kmeans_temp.inertia_)
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
plt.ylabel('Inertia (Within-cluster sum of squares)', fontsize=12, fontweight='bold')
plt.title('Elbow Method - Optimal K', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Explanation:

- **inertia_**: Sum of squared distances to nearest cluster center.

- **Elbow point**: Where inertia decrease slows down (optimal K).

- **K_range(1, 11)**: Tests K from 1 to 10 clusters.

## Step 11: Save Results

In [ ]:
cluster_stats.to_csv(os.path.join('..', 'outputs', 'cluster_statistics.csv'))
pca_results = pd.DataFrame(pca_result[:, :5], columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5'])
pca_results.to_csv(os.path.join('..', 'outputs', 'pca_components.csv'), index=False)
print('Advanced analytics results saved!')
print('READY FOR NOTEBOOK 10 (Ensemble Methods)')

## Summary

Advanced statistical analysis:
1. T-test (Winter vs Summer comparison)
2. ANOVA (Regional differences)
3. Chi-square test (Category-Region dependence)
4. PCA (dimensionality reduction)
5. Scree plot (variance explanation)
6. PCA 2D visualization (seasonal clustering)
7. K-Means clustering (pollution profiles)
8. Elbow method (optimal cluster count)

**Next**: Notebook 10 - Ensemble Methods & Model Optimization